# Convert Gasperini at-scale dataset to MuData

Downloaded from https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE120861

In [1]:
# !pip install pyensembl
#!pip install openpyxl
# !pip install fast_matrix_market
!pyensembl install --release 75 --species human

2023-08-28 15:47:24,774 - pyensembl.shell - INFO - Running 'install' for EnsemblRelease(release=75, species='homo_sapiens')
2023-08-28 15:47:25,470 - pyensembl.sequence_data - INFO - Loaded sequence dictionary from /PHShome/ljb80/.cache/pyensembl/GRCh37/ensembl75/Homo_sapiens.GRCh37.75.cdna.all.fa.gz.pickle
2023-08-28 15:47:25,597 - pyensembl.sequence_data - INFO - Loaded sequence dictionary from /PHShome/ljb80/.cache/pyensembl/GRCh37/ensembl75/Homo_sapiens.GRCh37.75.ncrna.fa.gz.pickle
2023-08-28 15:47:25,978 - pyensembl.sequence_data - INFO - Loaded sequence dictionary from /PHShome/ljb80/.cache/pyensembl/GRCh37/ensembl75/Homo_sapiens.GRCh37.75.pep.all.fa.gz.pickle


In [2]:
import pandas as pd
import anndata as ad
import mudata as md
import numpy as np
import pandas as pd
from fast_matrix_market import mmread
from scipy.sparse import lil_matrix, csr_matrix

## Load scRNA data

In [3]:
data_dir = "/data/pinello/SHARED_DATA/gasperini_2019/gasperini_atscale"
prefix = "GSE120861_at_scale_screen"

In [4]:
!ls $data_dir

cols.txt
GSE120861_at_scale_screen.cds.rds
GSE120861_at_scale_screen.cells.txt.gz
GSE120861_at_scale_screen_chunk.00.h5mu
GSE120861_at_scale_screen_chunk.00.results.csv.gz
GSE120861_at_scale_screen_chunk.00.umi_counts.png
GSE120861_at_scale_screen_chunk.01.h5mu
GSE120861_at_scale_screen_chunk.01.results.csv.gz
GSE120861_at_scale_screen_chunk.01.umi_counts.png
GSE120861_at_scale_screen_chunk.02.h5mu
GSE120861_at_scale_screen_chunk.02.results.csv.gz
GSE120861_at_scale_screen_chunk.02.umi_counts.png
GSE120861_at_scale_screen_chunk.03.h5mu
GSE120861_at_scale_screen_chunk.03.results.csv.gz
GSE120861_at_scale_screen_chunk.03.umi_counts.png
GSE120861_at_scale_screen_chunk.04.h5mu
GSE120861_at_scale_screen_chunk.04.results.csv.gz
GSE120861_at_scale_screen_chunk.04.umi_counts.png
GSE120861_at_scale_screen_chunk.h5mu
GSE120861_at_scale_screen.covariates.csv.gz
GSE120861_at_scale_screen.exprs.mtx.gz
GSE120861_at_scale_screen.genes.txt.gz
GSE120861_at_scale_screen.h5mu
GSE120861_at_scale_screen.ph

### Load counts matrix as AnnData

In [5]:
%%time
adata = ad.AnnData(csr_matrix(mmread(f"{data_dir}/{prefix}.exprs.mtx.gz", parallelism=8).T, dtype=np.float32))
adata

CPU times: user 1min 45s, sys: 8.1 s, total: 1min 53s
Wall time: 55.9 s


AnnData object with n_obs × n_vars = 207324 × 13135

### Load genes and metadata

In [6]:
var_names = pd.read_table(f"{data_dir}/{prefix}.genes.txt.gz", header=None, names=['gene_id']).set_index('gene_id')
var_names

""
gene_id
ENSG00000238009
ENSG00000237683
ENSG00000228463
ENSG00000237094
ENSG00000235373
...
ENSG00000215689
ENSG00000215781
ENSG00000220023


In [7]:
import pyensembl
pyensembl.__version__
ensembl = pyensembl.EnsemblRelease(release=75)

genes = [ensembl.gene_by_id(gene_id) for gene_id in var_names.index]
adata.var = pd.DataFrame(
    {
        'gene_name': [g.gene_name for g in genes],
        'contig': [g.contig for g in genes],
        'start': [g.start for g in genes],
        'end': [g.end for g in genes]
    },
    index = var_names.index
)
adata.var

,gene_name,contig,start,end
gene_id,,,,
ENSG00000238009,RP11-34P13.7,1,89295,133566
ENSG00000237683,AL627309.1,1,134901,139379
ENSG00000228463,AP006222.2,1,227615,267253
ENSG00000237094,RP4-669L17.10,1,317720,453948
ENSG00000235373,RP11-206L10.3,1,677193,685396
...,...,...,...,...
ENSG00000215689,MGC39584,GL000193.1,49252,88375
ENSG00000215781,AC011043.1,GL000195.1,43717,74342
ENSG00000220023,AL592183.1,GL000219.1,53831,99642


### Load cell metadata

Note: metadata column names were manually copied over from monocle .rds file since they're missing in this table

In [8]:
obs = pd.read_table(
    f"{data_dir}/{prefix}.phenoData.txt.gz",
    header = None,
    sep=" ",
    names=["sample", "cell", "total_umis", "Size_Factor", "gene", "all_gene",
           "barcode", "read_count", "umi_count", "proportion", "guide_count",
           "sample_directory", "ko_barcode_file", "id", "prep_batch",
           "within_batch_chip", "within_chip_lane", "percent.mito"
          ]
)

obs = obs.set_index("cell")
obs

,sample,total_umis,Size_Factor,gene,all_gene,barcode,read_count,umi_count,proportion,guide_count,sample_directory,ko_barcode_file,id,prep_batch,within_batch_chip,within_chip_lane,percent.mito
cell,,,,,,,,,,,,,,,,,
AAACCTGAGAGGTACC-1_1A_1_SI-GA-E2,1A_1_SI-GA-E2,17572.0,1.009682,chr10.845_top_two_chr1.11183_top_two_chr1.1129...,chr10.845_top_two_chr1.11183_top_two_chr1.1129...,AGAAAGCTCCTCCAGTTCAC_TGATCGCTTTGACTGTGACA_ACAA...,14135.0,964.0,0.969819,67.0,1A_1_SI-GA-E2,guide_libraries/1A_1.gRNAcaptured.txt,1A_1,prep_batch_1,within_batch_chip_A,within_chip_lane_1,0.058787
AAACCTGAGTCAATAG-1_1A_1_SI-GA-E2,1A_1_SI-GA-E2,8923.0,0.939677,chr1.12695_top_two_chr11.3294_top_two_chr1.679...,chr1.12695_top_two_chr11.3294_top_two_chr1.679...,GTAGAGCCTCCAGAACTGTG_AGGTTTATCCAGATGAACTG_CATC...,4329.0,293.0,0.844380,26.0,1A_1_SI-GA-E2,guide_libraries/1A_1.gRNAcaptured.txt,1A_1,prep_batch_1,within_batch_chip_A,within_chip_lane_1,0.036087
AAACCTGCAAACAACA-1_1A_1_SI-GA-E2,1A_1_SI-GA-E2,14637.0,0.990803,ALDH1A2_TSS_BRI3_TSS_chr10.1918_top_two_chr10....,ALDH1A2_TSS_BRI3_TSS_chr10.1918_top_two_chr10....,CCAAGGCGTCCTCAGACCAG_AGCTCCAGGAAGGACCCCCG_TCAC...,12362.0,884.0,0.950538,61.0,1A_1_SI-GA-E2,guide_libraries/1A_1.gRNAcaptured.txt,1A_1,prep_batch_1,within_batch_chip_A,within_chip_lane_1,0.069823
AAACCTGCACTTCTGC-1_1A_1_SI-GA-E2,1A_1_SI-GA-E2,22798.0,1.036578,C16orf91_TSS_chr1.11332_top_two_chr1.1933_top_...,C16orf91_TSS_chr1.11332_top_two_chr1.1933_top_...,GGCGTCAGTCGAGGAGTCAG_GCCAGCACTTCAGCTCACCG_GCTG...,7459.0,544.0,0.939551,39.0,1A_1_SI-GA-E2,guide_libraries/1A_1.gRNAcaptured.txt,1A_1,prep_batch_1,within_batch_chip_A,within_chip_lane_1,0.026187
AAACCTGCATGTAGTC-1_1A_1_SI-GA-E2,1A_1_SI-GA-E2,10136.0,0.952844,chr10.185_top_two_chr10.484_top_two_chr11.4167...,chr10.185_top_two_chr10.484_top_two_chr11.4167...,ATAAGGCACTCACATCCACC_GCTTGTCCCTAACACTCAGA_GGGC...,14831.0,1054.0,0.959927,37.0,1A_1_SI-GA-E2,guide_libraries/1A_1.gRNAcaptured.txt,1A_1,prep_batch_1,within_batch_chip_A,within_chip_lane_1,0.007991
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGTCAGTACCTACA-1_2B_8_SI-GA-H9,2B_8_SI-GA-H9,17938.0,1.011811,BRIX1_TSS_chr10.2059_top_two_chr10.350_top_two...,BRIX1_TSS_chr10.2059_top_two_chr10.350_top_two...,AGGAGGCCAAGAGCGCGGGG_TGATTGAAGGAGGCTCCCCA_GGTA...,4570.0,394.0,0.856522,31.0,2B_8_SI-GA-H9,guide_libraries/2B_8.gRNAcaptured.txt,2B_8,prep_batch_2,within_batch_chip_B,within_chip_lane_8,0.060598
TTTGTCAGTATCACCA-1_2B_8_SI-GA-H9,2B_8_SI-GA-H9,16543.0,1.003448,chr10.221_top_two_chr11.3853_top_two_chr11.498...,chr10.221_top_two_chr11.3853_top_two_chr11.498...,CCTGCAACTGTCTATGGCCT_TCAGTGGGTGAGTCTTCAGG_TGAG...,4311.0,387.0,0.861915,33.0,2B_8_SI-GA-H9,guide_libraries/2B_8.gRNAcaptured.txt,2B_8,prep_batch_2,within_batch_chip_B,within_chip_lane_8,0.057789
TTTGTCAGTTCAGACT-1_2B_8_SI-GA-H9,2B_8_SI-GA-H9,15009.0,0.993396,chr12.3400_top_two_chr4.1281_second_two_ID3_TSS,chr12.3400_top_two_chr4.1281_second_two_ID3_TSS,CCAGTTGCTAGGCAGGACAG_CTTTCAGGCAGGAATCTGAG_TGGT...,669.0,57.0,0.876923,3.0,2B_8_SI-GA-H9,guide_libraries/2B_8.gRNAcaptured.txt,2B_8,prep_batch_2,within_batch_chip_B,within_chip_lane_8,0.049037


In [9]:
adata.obs = obs.drop(["gene", "all_gene", "barcode", "sample_directory", "ko_barcode_file"], axis=1)


## Load gRNA data

In [10]:
pd.read_table(f"{data_dir}/GSE120861_grna_groups.at_scale.txt.gz", names=["element", "guide"])

grna_pairs = pd.read_table(f"{data_dir}/GSE120861_grna_groups.at_scale.txt.gz", names=["element", "guide"])
grna_pairs = grna_pairs.assign(guide19 = lambda x: x['guide'].str.slice(0,19))
grna_pairs
# grna_pairs

,element,guide,guide19
0,SH3BGRL3_TSS,AAACCGCTCCCGAGCACGGG,AAACCGCTCCCGAGCACGG
1,MTRNR2L8_TSS,AAATAGTGGGAAGATTCGTG,AAATAGTGGGAAGATTCGT
2,FAM83A_TSS,AACACACCACGGAGGAGTGG,AACACACCACGGAGGAGTG
3,ZNF593_TSS,AACAGCCCGGCCGGCCAAGG,AACAGCCCGGCCGGCCAAG
4,ATPIF1_TSS,AACGAGAGACTGCTTGCTGG,AACGAGAGACTGCTTGCTG
...,...,...,...
13184,random_5,TTAATTCCTCTGGCGCCGCT,TTAATTCCTCTGGCGCCGC
13185,random_14,TTAGCTTTGAAAACACACAC,TTAGCTTTGAAAACACACA
13186,random_2,TTATCTCTATTTGACAGACG,TTATCTCTATTTGACAGAC
13187,scrambled_23,TTGGGAATGCGAGGCAAAGG,TTGGGAATGCGAGGCAAAG


In [11]:
# get metadata from from supplement
grna_var = pd.read_excel(f"{data_dir}/mmc2.xlsx", sheet_name=1, dtype=str)
grna_var=grna_var.assign(guide19 = lambda x: x['Spacer'].str.slice(0,19))
grna_var=grna_var.set_index('guide19')

grna_var = grna_var.join(grna_pairs.set_index('guide19')).set_index('guide')
grna_var

,Spacer,Target_Site,chr.candidate_enhancer,start.candidate_enhancer,stop.candidate_enhancer,Category,element
guide,,,,,,,
AATGAGGAGCAAACGAAAAT,AATGAGGAGCAAACGAAAAT,control,NaN,NaN,NaN,NTC,scrambled_20
ACGAAATGTTTCATGACCAA,ACGAAATGTTTCATGACCAA,control,NaN,NaN,NaN,NTC,scrambled_23
ATAGATTTACGTTACTCTCT,ATAGATTTACGTTACTCTCT,control,NaN,NaN,NaN,NTC,scrambled_25
ATTAGCATCAGGTAGACTAA,ATTAGCATCAGGTAGACTAA,control,NaN,NaN,NaN,NTC,scrambled_18
CCATAAAGAATTCGGTGTAG,CCATAAAGAATTCGGTGTAG,control,NaN,NaN,NaN,NTC,scrambled_16
...,...,...,...,...,...,...,...
TCGGTGCGCGCCTGTCCGGG,TCGGTGCGCGCCTGTCCGG,ZNF593,NaN,NaN,NaN,TSS,ZNF593_TSS
AGAGTGCGGGCGGAAGACGG,AGAGTGCGGGCGGAAGACG,ZNHIT1,NaN,NaN,NaN,TSS,ZNHIT1_TSS
AGTTGTGTTGTGCCAATGGG,AGTTGTGTTGTGCCAATGG,ZNHIT1,NaN,NaN,NaN,TSS,ZNHIT1_TSS


In [26]:
grna_var.value_counts("Category")

Category
candidate_enhancer:picked_by_model_built_from_pilot              7706
candidate_enhancer:repeated_from_pilot:top_gRNA_pair             1956
candidate_enhancer:picked_by_exploratory_submodular_selection    1896
TSS                                                               762
candidate_enhancer:repeated_from_pilot:alternative_gRNA_pair      754
NTC                                                               101
Positive_control_to_globin_locus                                   14
dtype: int64

In [12]:
grna_var['chr.candidate_enhancer'].unique()

array([nan, 'chr10', 'chr1', 'chr11', 'chr12', 'chr13', 'chr14', 'chr15',
       'chr16', 'chr17', 'chr18', 'chr19', 'chr20', 'chr2', 'chr21',
       'chr22', 'chr3', 'chr4', 'chr5', 'chr6', 'chr7', 'chr8', 'chr9',
       'chrX'], dtype=object)

### Load gRNA targeting info

### Convert guides to sparse matrix
Perturbations per cell are stored as underscore-separated barcodes, want to convert to a sparse matrix for analysis

In [13]:
obs['barcode'].iloc[0]

'AGAAAGCTCCTCCAGTTCAC_TGATCGCTTTGACTGTGACA_ACAATAAAGAACAGAACACA_GTAAATTGAGACCTCAGGAG_TCTTCCCCCCACCAATAACA_GAGAAAAAAACAATTCAGGC_TCTTAGAGTTCACAGAAGAA_GCTGGGAATTTCTCTCCTGG_AGTGTAACAGAATATCAAAT_ACCCACTGTGACTAGACAAA_AGAAGGATAGAGACTGCTGG_CCAGGCACTTGTGAGAACAA_TGATGGTGTCCCCACCCAAA_GCAGGCCCCATGGATACCCG_AAGGAGTGTGTTCCACACCA_GTACCCTCCCTACCCCCGAG_TGCCTGCTAGAGTCAATAGG_AATGCCAGTTTCCCCCACAG_GCCATTGCTGTAGAGACACT_GCCCAGTCAGAACCCAGGAA_GCACAGATTTACACGCCCGT_GAGAGGCTGCCAGCCCACAG_TGTTCTATATTGCCACCTAG_AGCATCAAATTGCAGAGCAG_ACGAAGAATGAATTGAAGGG_CTGTTTCAGAAAGCTCCCAA_ACTTTGAGCTGCTTCAAGGG_GAGACCTTCCCCCTACCCAG_TTTCCCCGCTGACAGACTGA_ATGACTGCCCCCAGCAGCAA_GGCGCAAAGACAGTGCCAGA_CTCTGACTCACACAACAGGA_GCAAGTTTGCTTTCTCCTGG_TTGAAAGACACATAGCACGA_TCACAGATATTGACTGCCCT_ACAACCCCAAGAACTAGCGG_GCAGCGAAGCTGTTCCACCA_TGCTGCATCCAGATGTTACG_AGACACAGTCAAATGAGGCA_AGCTCTGTCAACCTGCCATG_TCTGCCTGAATGTTTCTCAG_CTGTTTATACCGAGCAGTAG_TGAGCTCCGCCTACACACGG_TTAGTAGAGTGTAGACTGGG_CTCCTTACCCCAGCCAATCG_GGGCCATTACCTTTGCAGAG_GGAGCCCACGACGCTCAAGG_GTTTCCTTATGT

In [ ]:

df = obs

# Step 0: helper function for splitting barcodes
def split_row(row):
    if type(row) == str:
        barcodes = row.split('_')
    else:
        barcodes = []
    return barcodes

# Step 1: Create a barcode-to-index mapping
barcode_to_index = {g:i for i, g in enumerate(grna_var.index)}

# Step 2: Create a sparse matrix using lil_matrix
num_samples = len(df)
num_barcodes = len(barcode_to_index)
sparse_matrix = lil_matrix((num_samples, num_barcodes), dtype=np.float32)

for i, row in enumerate(df['barcode']):
    barcodes = split_row(row)
    for barcode in barcodes:
        barcode_index = barcode_to_index[barcode]
        sparse_matrix[i, barcode_index] = 1

# Convert to a more efficient format if needed (e.g., csr_matrix)
sparse_matrix = sparse_matrix.tocsr()
sparse_matrix

### Put relevant fields in the gRNA adata

In [ ]:
gdata = ad.AnnData(sparse_matrix, dtype=np.float32)
gdata.obs_names=adata.obs_names
gdata.var = grna_var
gdata

In [ ]:

guide_by_element = pd.get_dummies(gdata.var["element"])
gdata.varm["element_targeted"]=csr_matrix(guide_by_element.values.astype(np.bool_))
gdata.uns["elements"] = list(guide_by_element.columns)
# pd.get_dummies(gdata.var["element"], sparse=True)
# gdata

## Export as MuData

In [ ]:
mdata = md.MuData({"rna":adata, "grna":gdata})

In [ ]:
mdata['rna'].obs['sample']

In [ ]:
mdata.write(f"{data_dir}/GSE120861_at_scale_screen.h5mu")

## optional: test reading

In [ ]:
md.read(f"{data_dir}/GSE120861_at_scale_screen.h5mu")

In [ ]:
mdata['rna'].X

In [ ]:
tss_guides = mdata['grna'].var.query("Category=='TSS' or Category=='NTC'")
guide_cts = tss_guides.value_counts('Target_Site')
guide_cts
# guide_cts[guide_cts==2]